## Inicialización del entorno

### Configurar y montar Google Drive

In [1]:
import sys
from google.colab import drive

drive.mount('/content/drive')

ruta_base = '/content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/'

# Añadir la ruta al sistema
sys.path.append(ruta_base)

Mounted at /content/drive


### Importación de librerías

In [2]:
!pip install -U pymgrid
!pip install optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 91.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.1 MB/s eta 0:00:00
  Created wheel for pymgrid: filename=pymgrid-1.2.2-py3-none-any.whl size=3492845 sha256=a028ee88ccf40fab4197bccb6000afed135b2b4605710f2a61850274d1228152
  Stored in directory: /root/.cache/pip/wheels/22/74/ee/35d2b7ad23381dc521b2c8670c6e30be0b2e9fe80a1fe03160
Successfully built pymgrid
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 20.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from pymgrid import Microgrid
from pymgrid.modules import GridModule, BatteryModule, LoadModule, RenewableModule
import random
from collections import defaultdict

from custom_env_tabular import CustomEnvTabular

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import os
import pickle

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Carga y preparación de datos

In [4]:
# PRECIOS DE LA RED (Península 2025)
ruta_precios = ruta_base + 'data/precio2025-peninsula.csv'
df_precios = pd.read_csv(ruta_precios, sep=';')
df_precios['datetime'] = pd.to_datetime(df_precios['datetime'], utc=True)
df_precios = df_precios.sort_values('datetime').reset_index(drop=True)
# Convertir de €/MWh a €/kWh
precios_kwh = df_precios['value'].values / 1000.0

# DEMANDA (LOAD) Y GENERACIÓN SOLAR (PV)
ruta_load = ruta_base + 'data/pymgrid_data/load/RefBldgFullServiceRestaurantNew2004_v1.3_7.1_6A_USA_MN_MINNEAPOLIS.csv'
ruta_pv = ruta_base + 'data/pymgrid_data/pv/SanFrancisco_724940TYA.csv'

df_load = pd.read_csv(ruta_load)
df_pv = pd.read_csv(ruta_pv)

load_series = df_load.iloc[:, -1].values
pv_series = df_pv.iloc[:, -1].values

# Asegurarnos de que todas las series tienen la misma longitud (evitar desajustes si un año es bisiesto, etc.)
min_len = min(len(precios_kwh), len(load_series), len(pv_series), 8760)
precios_kwh = precios_kwh[:min_len]
load_series = load_series[:min_len]
pv_series = pv_series[:min_len]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Creación de los módulos de pymgrid

In [5]:
# RED ELÉCTRICA (GridModule)
# En las versiones recientes de pymgrid, los datos de la red se pasan como un DataFrame
grid_ts = pd.DataFrame({
    'import_price': precios_kwh,
    'export_price': precios_kwh * 0.5,
    'co2_per_kwh': 0.0                 # Rellenamos con 0, pero pymgrid necesita esta columna
})

grid = GridModule(
    max_import=10000.0,
    max_export=10000.0,
    time_series=grid_ts
)

# BATERÍA (BatteryModule)
battery = BatteryModule(
    min_capacity=10.0,
    max_capacity=200.0,
    max_charge=50.0,
    max_discharge=50.0,
    efficiency=0.9,
    init_soc=0.5
)

# DEMANDA Y PLACAS SOLARES
# Les pasamos directamente los arrays con los kW de cada hora
load = LoadModule(time_series=load_series)
pv = RenewableModule(time_series=pv_series)

### Ensamblaje de la microrred y el entorno gymnasium

In [6]:
# Creamos la lista de tuplas con el nombre ('string') y el objeto de cada módulo
modules = [
    ('grid', grid),
    ('battery', battery),
    ('load', load),
    ('pv', pv)
]

# Construimos la microrred
microrred = Microgrid(modules)

# La conectamos a vuestro entorno
env = CustomEnvTabular(
    pymgrid_network=microrred,
    horizon=min_len,
    low_soc_penalty=10.0
)

### Prueba de funcionamiento

In [7]:
obs, info = env.reset()
print("¡Entorno inicializado con éxito, sin errores!")
print(f"Estado Inicial Discretizado [Carga_Neta, Batería_SoC, Precio, Hora]: {obs}")
print(f"Precio extraído para la primera hora: {info['current_import_price']:.4f} €/kWh")

¡Entorno inicializado con éxito, sin errores!
Estado Inicial Discretizado [Carga_Neta, Batería_SoC, Precio, Hora]: [2 5 3 0]
Precio extraído para la primera hora: 0.1832 €/kWh


# Funciones auxiliares

In [8]:
# 1. FUNCIÓN DE MUESTREO DE HIPERPARÁMETROS
def sample_q_params(trial):
    """Muestrea los hiperparámetros para Q-Learning Tabular."""
    return {
        "alpha": trial.suggest_float("alpha", 0.01, 0.3, log=True),
        "gamma": trial.suggest_float("gamma", 0.90, 0.999),
        "epsilon_decay_step": trial.suggest_float("epsilon_decay_step", 0.99990, 0.99999),
        "epsilon_min": 0.05
    }

In [9]:
# 2. FUNCIÓN DE EVALUACIÓN PURA (Sin exploración)
def evaluate_agent(env, Q, eval_episodes=1):
    """Evalúa la Q-Table actual (pura explotación) y devuelve el coste medio."""
    total_eval_cost = 0
    for _ in range(eval_episodes):
        obs, info = env.reset()
        state = tuple(obs)
        done = False
        ep_cost = 0

        while not done:
            # 1. Elegir la acción (ya sea por Q-Table o al azar si es desconocido)
            if state in Q:
                q_values = Q[state]
                # Encuentra los valores máximos y elige uno al azar entre los empatados
                best_actions = np.where(q_values == np.max(q_values))[0]
                action = np.random.choice(best_actions)
            else:
                action = env.action_space.sample()

            # 2. Ejecutar la acción en el entorno
            next_obs, reward, terminated, truncated, info = env.step(action)
            state = tuple(next_obs)
            done = terminated or truncated
            ep_cost += info.get('cost', 0)

        total_eval_cost += ep_cost

    return total_eval_cost / eval_episodes

In [10]:
# Directorio global para guardar la mejor Q-Table
GLOBAL_BEST_DIR = "/content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/"
os.makedirs(GLOBAL_BEST_DIR, exist_ok=True)
GLOBAL_BEST_PATH = os.path.join(GLOBAL_BEST_DIR, "best_q_table.pkl")
DB_PATH = "sqlite:////content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/optuna_microgrid.db"


try:
    # Si ya hay un estudio guardado, recuperamos su mejor coste
    temp_study = optuna.load_study(study_name="microgrid_qlearning_v1", storage=DB_PATH)
    global_best_cost = temp_study.best_value
    print(f"Reanudando sesión. Mejor coste histórico protegido: {global_best_cost:.2f} €")
except Exception:
    # Si es la primera vez que lo ejecutamos
    global_best_cost = float('inf')
    print("Iniciando nuevo estudio desde cero. global_best_cost = inf")

Iniciando nuevo estudio desde cero. global_best_cost = inf


In [11]:
# 3. FUNCIÓN OBJECTIVE CON PRUNING Y TRACKER GLOBAL
def objective(trial: optuna.Trial) -> float:
    global global_best_cost

    # Instanciar entornos limpios para este trial
    microrred = Microgrid([('grid', grid), ('battery', battery), ('load', load), ('pv', pv)])
    env = CustomEnvTabular(pymgrid_network=microrred, horizon=min_len, low_soc_penalty=10.0)

    microrred_eval = Microgrid([('grid', grid), ('battery', battery), ('load', load), ('pv', pv)])
    eval_env = CustomEnvTabular(pymgrid_network=microrred_eval, horizon=min_len, low_soc_penalty=10.0)

    # 1. Obtener hiperparámetros
    params = sample_q_params(trial)
    alpha = params["alpha"]
    gamma = params["gamma"]
    epsilon_decay_step = params["epsilon_decay_step"]
    epsilon_min = params["epsilon_min"]

    # 2. Inicializar Q-Table y variables
    Q = defaultdict(lambda: np.zeros(env.action_space.n))
    epsilon = 1.0
    # Al tener 8760 pasos por episodio, 10 episodios son 87.600 actualizaciones de la Q-Table. suficiente!
    n_episodes = 10
    eval_freq = 2  # Evaluar y reportar a Optuna cada 2 episodios

    # 3. Bucle de Entrenamiento
    for ep in range(n_episodes):
        obs, info = env.reset()
        state = tuple(obs)
        done = False

        while not done:
            if random.uniform(0, 1) < epsilon:
                action = env.action_space.sample()
            else:
                # Si el estado existe y tiene valores, elegimos el mejor (desempatando al azar)
                if state in Q:
                    q_values = Q[state]
                    best_actions = np.where(q_values == np.max(q_values))[0]
                    action = np.random.choice(best_actions)
                # Si es un estado completamente nuevo, elegimos al azar
                else:
                    action = env.action_space.sample()

            next_obs, reward, terminated, truncated, info = env.step(action)
            next_state = tuple(next_obs)
            done = terminated or truncated

            best_next_action = np.argmax(Q[next_state])
            Q[state][action] += alpha * (reward + gamma * Q[next_state][best_next_action] - Q[state][action])
            state = next_state

        epsilon = max(epsilon_min, epsilon * epsilon_decay_step)

        # -------------------------------------------------------------------
        # REPORTE A OPTUNA Y PRUNING (EVAL_CALLBACK manual)
        # -------------------------------------------------------------------
        if (ep + 1) % eval_freq == 0:
            current_cost = evaluate_agent(eval_env, Q)

            # Reportar a Optuna el coste actual en este paso intermedio
            trial.report(current_cost, ep)

            # Comprobar si debemos cortar este trial (Pruning)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            # Comprobar si es el mejor modelo global histórico
            if current_cost < global_best_cost:
                global_best_cost = current_cost
                # Guardar la Q-Table como un diccionario normal para que pickle no falle
                with open(GLOBAL_BEST_PATH, 'wb') as f:
                    pickle.dump(dict(Q), f)
                print(f"NUEVO GLOBAL BEST: {current_cost:.2f} € → guardado en {GLOBAL_BEST_PATH}")

    # Devolver el coste final del agente entrenado
    return evaluate_agent(eval_env, Q)

# Búsqueda de mejores hiperparámetros

In [ ]:
sampler = TPESampler(n_startup_trials=5)
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=2)

study = optuna.create_study(
    study_name="microgrid_qlearning_v1",
    storage="sqlite:////content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/optuna_microgrid.db",
    direction="minimize", # Minimizar coste en €
    sampler=sampler,
    pruner=pruner,
    load_if_exists=True
)

try:
    study.optimize(objective, n_trials=30, n_jobs=1)
except KeyboardInterrupt:
    print("Búsqueda interrumpida manualmente.")

print("\nNumber of finished trials: ", len(study.trials))
print("Best trial:")
best_trial = study.best_trial
print(f"  Coste Medio: {best_trial.value:.2f} €")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-03-21 19:48:32,366] A new study created in RDB with name: microgrid_qlearning_v1


NUEVO GLOBAL BEST: 1479341.91 € → guardado en /content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/best_q_table.pkl
NUEVO GLOBAL BEST: 1358259.79 € → guardado en /content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/best_q_table.pkl
NUEVO GLOBAL BEST: 1273304.88 € → guardado en /content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/best_q_table.pkl
NUEVO GLOBAL BEST: 1221481.83 € → guardado en /content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/best_q_table.pkl
NUEVO GLOBAL BEST: 1208348.81 € → guardado en /content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/best_q_table.pkl


[I 2026-03-21 19:53:34,926] Trial 0 finished with value: 1198141.8417531587 and parameters: {'alpha': 0.049101375098945396, 'gamma': 0.9432341961743936, 'epsilon_decay_step': 0.9999527002502668}. Best is trial 0 with value: 1198141.8417531587.


NUEVO GLOBAL BEST: 1178193.29 € → guardado en /content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/Q-learning/best_q_table.pkl


[I 2026-03-21 19:58:34,774] Trial 1 finished with value: 1184685.2732756962 and parameters: {'alpha': 0.10550341772758057, 'gamma': 0.9375132277214677, 'epsilon_decay_step': 0.9999654713586696}. Best is trial 1 with value: 1184685.2732756962.
[I 2026-03-21 20:03:34,661] Trial 2 finished with value: 1222315.1404707527 and parameters: {'alpha': 0.014298900436033563, 'gamma': 0.9392108179939912, 'epsilon_decay_step': 0.9999104394516609}. Best is trial 1 with value: 1184685.2732756962.
[I 2026-03-21 20:08:34,513] Trial 3 finished with value: 1211971.1939191204 and parameters: {'alpha': 0.024932037447412733, 'gamma': 0.9814311502781274, 'epsilon_decay_step': 0.9999749157160954}. Best is trial 1 with value: 1184685.2732756962.
[I 2026-03-21 20:13:36,856] Trial 4 finished with value: 1202815.151949313 and parameters: {'alpha': 0.10266723389052537, 'gamma': 0.9733552278677815, 'epsilon_decay_step': 0.9999319287461748}. Best is trial 1 with value: 1184685.2732756962.
[I 2026-03-21 20:18:37,707]

# Entrenamiento